# Qwen3-8B: download and fit a J-Lens
This adapts your WikiText-103 fitting notebook. It fits the **35 supported source blocks** (0–34) to the final block 35. 

The Qwen model is not a COCONUT checkpoint; use its chat template in the analysis notebook.

## Download Wikitext103

In [ ]:
from pathlib import Path
from shutil import copy2
from huggingface_hub import hf_hub_download

DATA_DIR = Path(r"D:\Researches\InterpreteJLens\data\wikitext103")
DATA_DIR.mkdir(parents=True, exist_ok=True)

for i in range(2):
    name = f"train-{i:05d}-of-00002.parquet"
    cached = hf_hub_download(
        repo_id="Salesforce/wikitext",
        repo_type="dataset",
        filename=f"wikitext-103-v1/{name}",
    )
    destination = DATA_DIR / name
    copy2(cached, destination)
    print(destination)

## Download J-lens and register for it

In [ ]:
%pip install -U git+https://github.com/anthropics/jacobian-lens.git

## Download Qwen3-8B

In [ ]:
from pathlib import Path
import torch, transformers, jlens
from huggingface_hub import snapshot_download

ROOT = Path(r"D:\Researches\InterpreteJLens")
MODEL_DIR = ROOT / "base_models" / "Qwen3-8B"
OUT = ROOT / "jlens_outputs" / "qwen3_8b"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Qwen/Qwen3-8B"
print("torch", torch.__version__, "transformers", transformers.__version__)
print("jlens", getattr(jlens, "__file__", "unknown"))


In [ ]:
# Download the unquantized official weights, config and tokenizer (~17 GB plus cache).
# snapshot_download resumes interrupted transfers; local_dir is reusable offline.
model_path = snapshot_download(repo_id=MODEL_ID, local_dir=str(MODEL_DIR))
print(model_path)


## Fit J-lens

In [ ]:
# Full BF16 weights alone need ~16.4 GB. Jacobian fitting needs much more.
# Do not silently offload to CPU or fit an unrelated 4-bit model.
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this notebook.")
mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
if mem_gb < 40:
    raise RuntimeError(f"Found {mem_gb:.1f} GiB GPU; full Qwen3-8B J-Lens fitting needs a larger GPU (recommend >=48 GB).")

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.bfloat16, attn_implementation="sdpa",
).to("cuda").eval()
lens_model = jlens.from_hf(model, tokenizer, compile=False)
assert lens_model.n_layers == 36, lens_model.n_layers
source_layers = list(range(lens_model.n_layers - 1))
print(lens_model, "source layers:", source_layers[0], "..", source_layers[-1])


## 1000 prompts x 128 tokens is the setting for fitting J-lens proved by Authropic

In [ ]:
# Reuse your two local WikiText-103 train parquet files.

N_PROMPTS = 1000  # use 2 for the first smoke run, then choose a NEW checkpoint path
MAX_SEQ_LEN = 128

import random, pandas as pd, json
DATA_DIR = ROOT / "data" / "wikitext103"
paths = [DATA_DIR / f"train-{j:05d}-of-00002.parquet" for j in (0,1)]
for path in paths:
    if not path.is_file():
        raise FileNotFoundError(path)
texts = pd.concat([pd.read_parquet(path)["text"] for path in paths], ignore_index=True)
rng = random.Random(43)
texts = [t.strip() for t in texts if isinstance(t,str) and t.strip() and not t.strip().startswith("=")]
rng.shuffle(texts)
prompts = []
for t in texts:
    ids = tokenizer.encode(t, add_special_tokens=False)
    if len(ids) < 64:
        continue
    if len(ids) > MAX_SEQ_LEN:
        s = rng.randrange(len(ids) - MAX_SEQ_LEN + 1)
        ids = ids[s:s+MAX_SEQ_LEN]
    prompts.append(tokenizer.decode(ids, skip_special_tokens=True).strip())
    if len(prompts) >= N_PROMPTS:
        break
if len(prompts) != N_PROMPTS:
    raise RuntimeError(f"Only {len(prompts)} eligible calibration prompts")
prompt_file = OUT / f"wikitext103_prompts_{N_PROMPTS}_seed43.json"
prompt_file.write_text(json.dumps(prompts, ensure_ascii=False, indent=2), encoding="utf-8")
print(len(prompts), "calibration prompts saved:", prompt_file)


In [ ]:
# Full all-layer fitting is expensive: 4096/2 = 2048 backward passes PER prompt.
# dim_batch=2 trades speed for memory; increase only after checking peak memory.
# The checkpoint can exceed 2 GB. Keep enough free disk space.
fit_checkpoint = OUT / f"qwen3_8b_wikitext_{N_PROMPTS}_fit_checkpoint.pt"
lens_file = OUT / f"qwen3_8b_wikitext_{N_PROMPTS}_lens.pt"
lens = jlens.fit(
    lens_model, prompts=prompts, source_layers=source_layers,
    target_layer=lens_model.n_layers-1, dim_batch=2,
    max_seq_len=MAX_SEQ_LEN, skip_first=4,
    checkpoint_path=str(fit_checkpoint), checkpoint_every=10,
    resume=True,
)
lens.save(str(lens_file))
print("Fitted prompts:", lens.n_prompts, "source layers:", lens.source_layers)
print("Saved:", lens_file)
